In [1]:
import pandas as pd
import plotly.graph_objects as go

# file_path = "dados-tanque-cônico.csv"
file_path = "https://docs.google.com/spreadsheets/d/1UPq-_KwH0DZXkwSryfIc5ePeHTOPsQVSuMVaHeimwSI/export?format=csv&gid=0"

arquivo = pd.read_csv(file_path, decimal=",")
arquivo.head()

t = arquivo["t"].to_numpy()
Q = arquivo["Qi(t)"].to_numpy()
H = arquivo["H(t)"].to_numpy()


fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=Q, mode="lines", name="Q(t)"))
fig.add_trace(go.Scatter(x=t, y=H, mode="lines", name="H(t)"))
fig.update_layout(title="Gráfico de q e h em função de t", xaxis_title="t")


In [2]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

Q_interp = interp1d(t, Q, kind="linear")


def get_kp_tp(t):
    kp = 0.37
    tp = 50

    if t > 12:
        kp = 0.33
        tp = 51
    if t > 252:
        kp = 0.18
        tp = 10
    if t > 303:
        kp = 0.365
        tp = 20
    if t > 501:
        kp = 0.44
        tp = 60
    if t > 1002:
        kp = 0.55
        tp = 180
    if t > 2502:
        kp = 0.35
        tp = 180

    return kp, tp


def edo(t, y):
    H = y

    Q = Q_interp(t)
    kp, tp = get_kp_tp(t)

    dHdt = (kp * Q - H) / tp
    return [dHdt]


t_np = np.arange(t[0], t[-1], 0.1)

sol = solve_ivp(
    edo,
    [t[0], t[-1]],
    [H[0]],
    t_eval=t_np,
)


fig2 = go.Figure()
# fig2.add_trace(go.Scatter(x=t, y=Q, mode="lines", name="Q(t)"))
# fig2.add_trace(go.Scatter(x=t_np, y=Q_interp(t_np), mode="lines", name="Q(t) interpolado"))
fig2.add_trace(go.Scatter(x=t, y=H, mode="lines", name="H(t)"))
fig2.add_trace(go.Scatter(x=sol.t, y=sol.y[0], mode="lines", name="H(t) modelo"))
fig2.update_layout(title="Gráfico de q e h em função de t", xaxis_title="t")
